In [ ]:
!pip install -q mne scikit-learn joblib moabb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 980.1/980.1 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.5/257.5 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.2/120.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.

In [ ]:
# CELL 1 (FIXED): Cho2017 - Load, channel-select, filter, split, save

!pip install -q mne scikit-learn joblib moabb

import os
import numpy as np
import mne
import mne.channels
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from google.colab import drive

# Guarded montage patch (applied once).
if not getattr(mne.channels, '_montage_patched', False):
    _orig = mne.channels.make_standard_montage
    def _safe_make_standard_montage(kind, head_size='auto'):
        try:
            return _orig(kind, head_size=head_size)
        except KeyError:
            return _orig('standard_1020', head_size=head_size)
    mne.channels.make_standard_montage = _safe_make_standard_montage
    mne.channels._montage_patched = True
    try:
        import moabb.datasets.gigadb as _gigadb
        _gigadb.make_standard_montage = _safe_make_standard_montage
    except Exception:
        pass

from moabb.datasets import Cho2017
from moabb.paradigms import LeftRightImagery

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'
os.makedirs(CROSS_PROJECT_DIR, exist_ok=True)
print('CROSS_PROJECT_DIR:', CROSS_PROJECT_DIR)

dataset = Cho2017()
dataset.subject_list = [1]
paradigm = LeftRightImagery(fmin=1.0, fmax=100.0, tmin=0.0, tmax=3.0)
X_raw, y, metadata = paradigm.get_data(dataset=dataset, subjects=[1])
print('Raw X shape:', X_raw.shape)
print('Class distribution:', dict(zip(*np.unique(y, return_counts=True))))

# Cho2017 ships with the standard 64-channel 10-20 montage in fixed order.
CHO64_CHANNELS = [
    'Fp1','AF7','AF3','F1','F3','F5','F7','FT7','FC5','FC3','FC1','FCz','FC2','FC4','FC6','FT8',
    'Fp2','AF8','AF4','Fz','F2','F4','F6','F8','Cz','C1','C3','C5','T7','TP7','CP5','CP3',
    'CP1','CPz','CP2','CP4','CP6','TP8','T8','C2','C4','C6','P1','P3','P5','P7','P9','Pz',
    'P2','P4','P6','P8','P10','PO1','PO3','PO5','PO7','POz','PO2','PO4','PO6','PO8','Oz','O1',
]
# Fallback: infer from metadata columns if the list length mismatches.
if len(CHO64_CHANNELS) != X_raw.shape[1]:
    print('Channel list mismatch ({} vs {}), inferring from metadata.'.format(
        len(CHO64_CHANNELS), X_raw.shape[1]))
    CHO64_CHANNELS = [str(c) for c in metadata.columns[:X_raw.shape[1]]]
print('Number of channels:', len(CHO64_CHANNELS))

COMMON_CHANNELS = [
    'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
    'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
    'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
]
available = set(CHO64_CHANNELS)
missing = [ch for ch in COMMON_CHANNELS if ch not in available]
if missing:
    print('Missing channels (skipped):', missing)
    COMMON_CHANNELS = [ch for ch in COMMON_CHANNELS if ch in available]
print('Using {} channels: {}'.format(len(COMMON_CHANNELS), COMMON_CHANNELS))

ch_idx = [CHO64_CHANNELS.index(ch) for ch in COMMON_CHANNELS]
X_sub = X_raw[:, ch_idx, :].astype(np.float32)
print('After channel selection:', X_sub.shape)

sfreq = 512.0
bands = {'mu': (8.0, 12.0), 'beta': (13.0, 30.0)}
idx = np.arange(X_sub.shape[0])
idx_train, idx_test, y_train, y_test = train_test_split(
    idx, y, test_size=0.2, stratify=y, random_state=42,
)

for name, (lo, hi) in bands.items():
    b, a = butter(N=4, Wn=[lo, hi], btype='bandpass', fs=sfreq)
    Xf = filtfilt(b, a, X_sub, axis=-1).astype(np.float32)
    np.save(os.path.join(CROSS_PROJECT_DIR, 'cho_X_train_{}.npy'.format(name)), Xf[idx_train])
    np.save(os.path.join(CROSS_PROJECT_DIR, 'cho_X_test_{}.npy'.format(name)), Xf[idx_test])
    print('{} band: train {} test {}'.format(name, Xf[idx_train].shape, Xf[idx_test].shape))

np.save(os.path.join(CROSS_PROJECT_DIR, 'cho_y_train.npy'), y_train)
np.save(os.path.join(CROSS_PROJECT_DIR, 'cho_y_test.npy'), y_test)
print('Train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CROSS_PROJECT_DIR: /content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/


/tmp/ipykernel_1569/1736984083.py:19: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  return _orig(kind, head_size=head_size)


Raw X shape: (200, 64, 1537)
Class distribution: {np.str_('left_hand'): np.int64(100), np.str_('right_hand'): np.int64(100)}
Number of channels: 64
Using 17 channels: ['FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4']
After channel selection: (200, 17, 1537)
mu band: train (160, 17, 1537) test (40, 17, 1537)
beta band: train (160, 17, 1537) test (40, 17, 1537)
Train class counts: {np.str_('left_hand'): np.int64(80), np.str_('right_hand'): np.int64(80)}
Test class counts: {np.str_('left_hand'): np.int64(20), np.str_('right_hand'): np.int64(20)}
Saved to: /content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/


In [ ]:
# CELL 2: Cho2017 - FBCSP + QGA + SVM evaluation

import os
import numpy as np
import joblib
from mne.decoding import CSP
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
CROSS_PROJECT_DIR = '/content/drive/MyDrive/EEG_Quantum_Project_CrossDataset/'

BANDS = ['mu', 'beta']
N_COMP = 4

y_train = np.load(os.path.join(CROSS_PROJECT_DIR, 'cho_y_train.npy'), allow_pickle=True)
y_test = np.load(os.path.join(CROSS_PROJECT_DIR, 'cho_y_test.npy'), allow_pickle=True)

# --- FBCSP extraction (leak-free: CSP fit on training fold only) ---
csp_models = {}
train_feats, test_feats = [], []
for band in BANDS:
    Xtr = np.load(os.path.join(CROSS_PROJECT_DIR, 'cho_X_train_{}.npy'.format(band)))
    Xte = np.load(os.path.join(CROSS_PROJECT_DIR, 'cho_X_test_{}.npy'.format(band)))
    csp = CSP(n_components=N_COMP, reg=None, log=True, norm_trace=False)
    ftr = csp.fit_transform(Xtr, y_train)
    fte = csp.transform(Xte)
    csp_models[band] = csp
    train_feats.append(ftr)
    test_feats.append(fte)
    print('{} CSP -> train {} test {}'.format(band, ftr.shape, fte.shape))

X_train_fbcsp = np.concatenate(train_feats, axis=1).astype(np.float32)
X_test_fbcsp = np.concatenate(test_feats, axis=1).astype(np.float32)
print('Concatenated FBCSP train:', X_train_fbcsp.shape)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_fbcsp).astype(np.float32)
X_test = scaler.transform(X_test_fbcsp).astype(np.float32)

# ---------------------------------------------------------------------------
# QGA feature selection (identical algorithm to PhysioNetMI and BCIC-IV-2a)
# ---------------------------------------------------------------------------
N_FEATURES = X_train.shape[1]
POP_SIZE = 20
N_GEN = 30
MIN_FEATURES = 3
RNG = np.random.default_rng(42)
DELTA_THETA = 0.05 * np.pi

alpha = np.full((POP_SIZE, N_FEATURES), 1.0 / np.sqrt(2.0))
beta = np.full((POP_SIZE, N_FEATURES), 1.0 / np.sqrt(2.0))

def measure(a_row, b_row, rng):
    prob_one = b_row ** 2
    return (rng.random(N_FEATURES) < prob_one).astype(np.int32)

def fitness(mask):
    if mask.sum() < MIN_FEATURES:
        return 0.0
    Xs = X_train[:, mask.astype(bool)]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = SVC(kernel='rbf', class_weight='balanced', random_state=42)
    return float(cross_val_score(clf, Xs, y_train, cv=cv, scoring='f1_macro', n_jobs=-1).mean())

def rotate(a_row, b_row, x_current, b_best, current_fit, best_fit):
    for j in range(N_FEATURES):
        if x_current[j] == b_best[j]:
            continue
        sign = 1.0 if b_best[j] == 1 else -1.0
        if current_fit > best_fit:
            sign = -sign
        theta = sign * DELTA_THETA
        a_new = a_row[j] * np.cos(theta) - b_row[j] * np.sin(theta)
        b_new = a_row[j] * np.sin(theta) + b_row[j] * np.cos(theta)
        a_row[j] = a_new
        b_row[j] = b_new

best_mask, best_fit = None, -1.0
for gen in range(N_GEN):
    masks = np.array([measure(alpha[i], beta[i], RNG) for i in range(POP_SIZE)])
    fits = np.array([fitness(m) for m in masks])
    idx_best = int(np.argmax(fits))
    if fits[idx_best] > best_fit:
        best_fit = float(fits[idx_best])
        best_mask = masks[idx_best].copy()
    for i in range(POP_SIZE):
        rotate(alpha[i], beta[i], masks[i], best_mask, fits[i], best_fit)
    print('Gen {:3d} | best CV F1 {:.4f} | mean {:.4f} | size {}'.format(
        gen + 1, best_fit, fits.mean(), int(best_mask.sum())))

print('Final selected indices:', np.where(best_mask == 1)[0].tolist())
print('Final best CV macro F1: {:.4f}'.format(best_fit))

# ---------------------------------------------------------------------------
# SVM evaluation on full and QGA subsets
# ---------------------------------------------------------------------------
classes = ['left_hand', 'right_hand']
print('Baseline accuracy (chance): 0.5000')

param_grid = {'C': [0.1, 1.0, 10.0, 100.0, 1000.0],
              'gamma': ['scale', 0.01, 0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def fit_and_report(X_tr, X_te, tag):
    base = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    grid = GridSearchCV(base, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_tr, y_train)
    model = grid.best_estimator_
    y_pred = model.predict(X_te)
    print('--- {} ---'.format(tag))
    print('Best params:', grid.best_params_)
    print('CV f1_macro: {:.4f}'.format(grid.best_score_))
    print('Test accuracy: {:.4f}'.format(accuracy_score(y_test, y_pred)))
    print('Test macro F1: {:.4f}'.format(f1_score(y_test, y_pred, average='macro', zero_division=0)))
    print(classification_report(y_test, y_pred, labels=classes, digits=4, zero_division=0))
    print('Confusion matrix (rows=true, cols=pred):')
    print(confusion_matrix(y_test, y_pred, labels=classes))
    return model

full_model = fit_and_report(X_train, X_test, 'FULL FBCSP (8 feat) + SVM')
qga_model = fit_and_report(X_train[:, best_mask.astype(bool)], X_test[:, best_mask.astype(bool)],
                           'QGA-SELECTED ({} feat) + SVM'.format(int(best_mask.sum())))

joblib.dump(full_model, os.path.join(CROSS_PROJECT_DIR, 'cho_svm_full.joblib'))
joblib.dump(qga_model, os.path.join(CROSS_PROJECT_DIR, 'cho_svm_qga.joblib'))
np.save(os.path.join(CROSS_PROJECT_DIR, 'cho_qga_best_mask.npy'), best_mask)
joblib.dump(scaler, os.path.join(CROSS_PROJECT_DIR, 'cho_fbcsp_scaler.joblib'))
joblib.dump(csp_models, os.path.join(CROSS_PROJECT_DIR, 'cho_csp_models.joblib'))
print('Saved to:', CROSS_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Computing rank from data with rank=None
    Using tolerance 1.1e+03 (2.2e-16 eps * 17 dim * 2.9e+17  max singular value)
    Estimated rank (data): 17
    data: rank 17 computed from 17 data channels with 0 projectors
Reducing data rank from 17 -> 17
Estimating class=left_hand covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
mu CSP -> train (160, 4) test (40, 4)
Computing rank from data with rank=None
    Using tolerance 5.7e+02 (2.2e-16 eps * 17 dim * 1.5e+17  max singular value)
    Estimated rank (data): 17
    data: rank 17 computed from 17 data channels with 0 projectors
Reducing data rank from 17 -> 17
Estimating class=left_hand covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
beta CSP -> train (160, 4) test (40, 4)
Concatenated FBCSP train: (160, 8)
Gen   1 | best